## Example 1: RFC 3339 datetime string validation

In [ ]:
import rfc3339_validator

# Example 1: RFC 3339 datetime string validation
print("Example 1: RFC 3339 datetime string validation")

try:
    # Test various datetime formats
    test_datetimes = [
        '2023-12-25T10:30:00Z',           # Valid UTC
        '2023-12-25T10:30:00+05:00',      # Valid with timezone
        '2023-12-25T10:30:00.123Z',       # Valid with milliseconds
        '2023-12-25 10:30:00',            # Invalid (space instead of T)
        '2023-13-25T10:30:00Z',           # Invalid (month 13)
        '2023-12-25T25:30:00Z',           # Invalid (hour 25)
        '2023-12-25T10:30:00',            # Invalid (no offset: RFC 3339 requires Z or +hh:mm)
        '2023-12-25T10:30:00.123456Z',    # Valid with microseconds
        'invalid-datetime'                # Invalid format
    ]
    
    print("RFC 3339 datetime validation:")
    print("=" * 50)
    
    verdicts = []
    for dt_str in test_datetimes:
        is_valid = rfc3339_validator.validate_rfc3339(dt_str)
        verdicts.append(is_valid)
        status = "✓ VALID" if is_valid else "✗ INVALID"
        print(f"  {dt_str:30} -> {status}")

    assert verdicts == [True, True, True, False, False, False, False, True, False], verdicts
    valid_count = sum(verdicts)
    print(f"\nSummary: {valid_count}/{len(test_datetimes)} datetime strings are valid RFC 3339")
    
    print("rfc3339_validator example 1 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"rfc3339_validator example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 1 finished")

## Example 2: Time zone offsets

In [ ]:
import rfc3339_validator

# Example 2: Time zone offsets
print("Example 2: Time zone offsets")

try:
    cases = {
        '2023-12-25T10:30:00Z': True,
        '2023-12-25T10:30:00+05:30': True,
        '2023-12-25T10:30:00+23:59': True,     # the largest offset the grammar allows
        '2023-12-25T10:30:00-00:00': True,     # "-00:00" (offset unknown) is valid RFC 3339
        '2023-12-25T10:30:00+24:00': False,    # offset hours stop at 23
        '2023-12-25T10:30:00+0530': False,     # the offset needs a colon
        '2023-12-25T10:30:00': False,          # no offset at all
        '2023-12-25T10:30Z': False,            # seconds are required
    }
    for value, expected in cases.items():
        result = rfc3339_validator.validate_rfc3339(value)
        print(f"  {value:28} -> {result}")
        assert result is expected, (value, result)

    print("rfc3339_validator example 2 completed successfully")

except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"rfc3339_validator example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 2 finished")

## Example 3: Calendar checks: leap years and month lengths

In [ ]:
import rfc3339_validator

# Example 3: Calendar checks: leap years and month lengths
print("Example 3: Calendar checks: leap years and month lengths")

try:
    cases = {
        '2024-02-29T00:00:00Z': True,     # 2024 is a leap year
        '2000-02-29T00:00:00Z': True,     # divisible by 400: leap year
        '2023-02-29T00:00:00Z': False,    # 2023 is not
        '1900-02-29T00:00:00Z': False,    # divisible by 100 but not by 400: not a leap year
        '2023-04-31T00:00:00Z': False,    # April has 30 days
        '2023-12-31T23:59:59Z': True,
        '0000-01-01T00:00:00Z': False,    # year 0 is rejected
        '1990-12-31T23:59:60Z': False,    # leap seconds are not supported by this validator
    }
    for value, expected in cases.items():
        result = rfc3339_validator.validate_rfc3339(value)
        print(f"  {value:24} -> {result}")
        assert result is expected, (value, result)

    print("rfc3339_validator example 3 completed successfully")

except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"rfc3339_validator example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 3 finished")

## Example 4: Letter case and fractional seconds

In [ ]:
import rfc3339_validator

# Example 4: Letter case and fractional seconds
print("Example 4: Letter case and fractional seconds")

try:
    cases = {
        '2023-12-25T10:30:00.1Z': True,
        '2023-12-25T10:30:00.123456789Z': True,   # any number of fractional digits
        '2023-12-25T10:30:00.Z': False,           # a dot needs at least one digit
        '2023-12-25T10:30:00,5Z': False,          # a comma is not a decimal separator here
        # RFC 3339 allows lower-case 't' and 'z', but this validator only accepts upper case
        '2023-12-25t10:30:00Z': False,
        '2023-12-25T10:30:00z': False,
    }
    for value, expected in cases.items():
        result = rfc3339_validator.validate_rfc3339(value)
        print(f"  {value:32} -> {result}")
        assert result is expected, (value, result)

    print("rfc3339_validator example 4 completed successfully")

except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"rfc3339_validator example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 4 finished")

## Example 5: Input types and parsing validated values

In [ ]:
import rfc3339_validator

# Example 5: Input types and parsing validated values
print("Example 5: Input types and parsing validated values")

try:
    from datetime import datetime, timezone, timedelta

    # The validator only takes str: other types raise TypeError instead of returning False
    for value in (None, 20231225, b'2023-12-25T10:30:00Z'):
        try:
            rfc3339_validator.validate_rfc3339(value)
        except TypeError as err:
            print(f"  {value!r}: TypeError ({err})")
        else:
            raise AssertionError(f"{value!r} did not raise TypeError")

    # Validate first, then parse the accepted values with the standard library
    stamps = ['2023-12-25T10:30:00Z', 'yesterday', '2023-12-25T12:00:00+05:30', '2023-12-25 10:30:00Z']
    valid = [s for s in stamps if rfc3339_validator.validate_rfc3339(s)]
    parsed = [datetime.fromisoformat(s.replace('Z', '+00:00')) for s in valid]
    for s, dt in zip(valid, parsed):
        print(f"  {s} -> {dt.astimezone(timezone.utc).isoformat()}")
    assert valid == ['2023-12-25T10:30:00Z', '2023-12-25T12:00:00+05:30'], valid
    assert parsed[1].utcoffset() == timedelta(hours=5, minutes=30)
    assert parsed[1].astimezone(timezone.utc) == datetime(2023, 12, 25, 6, 30, tzinfo=timezone.utc)

    print("rfc3339_validator example 5 completed successfully")

except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"rfc3339_validator example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 5 finished")